# Composition at scale

### Overview

Does event composition differ between reviews, and is that difference organised by
anything (year, author, brand) at any unit of analysis?

**Input:** `checkpoints/event_sequences.jsonl` (the per-review event chains from 3.1) and
`data/label_to_nested_mapping_reference.csv`.
**Output:** `outputs_event_chains/experiment/composition_scale.csv`

**Pipeline:**
1. Setup, load, shared helpers.
2. Tier 1: does a single factor organise composition? (year / author / brand)
3. Tier 2: does brand organise it at a given time resolution? (brand x 3yr, brand x year)
4. Tier 3: within a brand, are its reviews internally coherent, and do they drift across their own history?
5. Assemble the rows, append to the stacked results file, and report.

## 1. Setup

In [1]:
# Cell 1: Imports

import json
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats
from scipy.sparse import csr_matrix
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt

In [2]:
# Cell 2: Parameters

CHECKPOINTS  = Path("checkpoints")
OUT_DIR      = Path("outputs_event_chains/experiment")
OUT_DIR.mkdir(exist_ok=True)

SEQ_PATH     = CHECKPOINTS / "event_sequences.jsonl"
NESTED_PATH  = Path("data/label_to_nested_mapping_reference.csv")
RESULTS_PATH = OUT_DIR / "composition_scale.csv"

nested_df       = pd.read_csv(NESTED_PATH)
LABEL_TO_MR     = dict(zip(nested_df["Label"], nested_df["MR_nested"]))
LABEL_TO_AGENCY = dict(zip(nested_df["Label"], nested_df["Agency_nested"]))


ALPHABETS = [
    ("Cluster_30", None),
    ("MR",         LABEL_TO_MR),
    ("Agency_6",   LABEL_TO_AGENCY),
]
BRAND_MIN = 24

PANEL_GROUPINGS = [
    (1, "year",   ["year"]),
    (1, "author", ["author"]),
    (1, "brand",  ["brand"]),
]

TIME_GROUPINGS = [
    (2, "brand x 3yr",  ["brand", "era"],  6),
    (2, "brand x year", ["brand", "year"], 2),
]

ERA_YEARS      = 3     # matched to the RollingLDA window
MIN_CHAIN_LEN  = 5
CLR_SMOOTHING  = 0.5
N_PERMUTATIONS = 200

# Pre-registered floors
OVERDISPERSION_FLOOR = 1.20
ETA_EXCESS_FLOOR     = 0.02

RANDOM_STATE = 42
RUN_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")

In [3]:
# Cell 3: Load once, relabel per alphabet

with open(SEQ_PATH) as f:
    sequences = [json.loads(line) for line in f]


_MISSING = {"", "nan", "none"}
sequences = [s for s in sequences if str(s.get("author")).strip().lower() not in _MISSING]

meta_all = pd.DataFrame([
    {"review_id": s["review_id"], "brand": s["brand"], "year": s["year"],
     "season": s["season"], "author": s["author"]}
    for s in sequences
])

_cleaned = pd.read_csv("data/stylecom_cleaned.csv")
_id2cal = dict(enumerate(pd.to_datetime(_cleaned["date"], format="%d-%b-%y").dt.year))
meta_all["year"] = meta_all["review_id"].map(_id2cal)

meta_all["era"] = (pd.to_numeric(meta_all["year"], errors="coerce") // ERA_YEARS) * ERA_YEARS

# brands with at least BRAND_MIN reviews
PANEL_BRANDS = set(meta_all["brand"].value_counts().loc[lambda s: s >= BRAND_MIN].index)
print(f"panel: {len(PANEL_BRANDS)} designers (brands with >= {BRAND_MIN} reviews)")


def panel_mask(meta):
    """Boolean mask selecting the reviews of the 65-designer panel."""
    return meta["brand"].isin(PANEL_BRANDS).to_numpy()


def build(remap, min_chain):
    """Count matrix + metadata for one alphabet."""
    chains, keep_idx = [], []
    for i, s in enumerate(sequences):
        toks = [e["event"] for e in s["events"]]
        if remap is not None:
            toks = [remap[t] for t in toks]
        if len(toks) >= min_chain:
            chains.append(toks)
            keep_idx.append(i)

    vocab = sorted({e for c in chains for e in c})
    idx   = {e: i for i, e in enumerate(vocab)}
    counts = np.zeros((len(chains), len(vocab)), dtype=np.int64)
    for i, c in enumerate(chains):
        for e in c:
            counts[i, idx[e]] += 1
    return counts, meta_all.iloc[keep_idx].reset_index(drop=True), vocab


DATA = {}
for name, remap in ALPHABETS:
    counts, meta, vocab = build(remap, MIN_CHAIN_LEN)
    DATA[name] = {"counts": counts, "meta": meta, "vocab": vocab}
    pm = panel_mask(meta)
    print(f"{name:14s} {counts.shape[0]:,} reviews   {len(vocab):2d} types   "
          f"median chain {int(np.median(counts.sum(axis=1)))}   "
          f"panel {int(pm.sum()):,} reviews")

DATA0 = {}
for name, remap in ALPHABETS:
    counts, meta, vocab = build(remap, 1)
    DATA0[name] = {"counts": counts, "meta": meta, "vocab": vocab}

panel: 65 designers (brands with >= 24 reviews)
Cluster_30     6,458 reviews   30 types   median chain 17   panel 1,883 reviews
MR             6,458 reviews    4 types   median chain 17   panel 1,883 reviews
Agency_6       6,458 reviews    6 types   median chain 17   panel 1,883 reviews


In [4]:
# Cell 4: Shared helpers

def clr(counts, smoothing=CLR_SMOOTHING):
    """Centred log-ratio, so Euclidean distance = Aitchison distance."""
    x  = counts + smoothing
    x  = x / x.sum(axis=1, keepdims=True)
    lx = np.log(x)
    return lx - lx.mean(axis=1, keepdims=True)


def group_key(meta, keys):
    return meta[keys].astype(str).agg(" | ".join, axis=1)


def apply_min(key, min_reviews):
    """Boolean mask of reviews in groups meeting the minimum."""
    sizes = key.value_counts()
    kept  = sizes[sizes >= min_reviews].index
    return key.isin(kept).to_numpy()


def ssb(X, labels, G, grand):
    """Between-group sum of squares, vectorised: sum_g n_g * ||mean_g - grand||^2."""
    n = X.shape[0]
    M = csr_matrix((np.ones(n), (labels, np.arange(n))), shape=(G, n))
    sums = M @ X
    cnt  = np.bincount(labels, minlength=G).astype(float)
    nz   = cnt > 0
    means = np.zeros_like(sums)
    means[nz] = sums[nz] / cnt[nz, None]
    return float((cnt[nz, None] * (means[nz] - grand) ** 2).sum())


def permanova(X, key_values, n_perm, seed):
    """PERMANOVA eta-squared against a group-label permutation null."""
    labels = pd.Categorical(key_values).codes.astype(np.int64)
    G = int(labels.max()) + 1
    grand = X.mean(axis=0)
    sst = float(((X - grand) ** 2).sum())

    obs = ssb(X, labels, G, grand)
    rng = np.random.default_rng(seed)
    null = np.array([ssb(X, rng.permutation(labels), G, grand) for _ in range(n_perm)])

    eta, eta_null = obs / sst, float((null / sst).mean())
    return {"n_groups": G, "n_reviews": int(X.shape[0]), "eta": eta,
            "eta_null": eta_null, "eta_excess": eta - eta_null}


def overdispersion(counts, n_perm, seed):
    """Observed variance of each type's share vs a multinomial null at the same sizes."""
    L = counts.sum(axis=1)
    keep = L > 0
    counts, L = counts[keep], L[keep]
    props = counts / L[:, None]
    p = counts.sum(axis=0) / counts.sum()
    obs = props.var(axis=0, ddof=1)

    rng = np.random.default_rng(seed)
    nv = np.empty((n_perm, counts.shape[1]))
    for i in range(n_perm):
        sim = np.vstack([rng.multinomial(int(n), p) for n in L])
        nv[i] = (sim / L[:, None]).var(axis=0, ddof=1)
    nm = nv.mean(axis=0)
    return {"overdisp_median": float(np.median(obs / nm)),
            "max_true_sd_pp": float(np.sqrt(np.maximum(obs - nm, 0)).max() * 100)}


results = []   # one dict per row; assembled in Section 6

## 2. Tiers 1 and 2: does any grouping organise composition?

Each review becomes a CLR vector; PERMANOVA asks how much variance the grouping explains (`eta`), and the null permutes group labels while holding group sizes fixed.

In [5]:
# Cell 6: Tiers 1 and 2: PERMANOVA

for tier, gname, keys in PANEL_GROUPINGS:
    for aname, _ in ALPHABETS:
        counts, meta = DATA[aname]["counts"], DATA[aname]["meta"]
        mask = panel_mask(meta)
        key  = group_key(meta[mask], keys).reset_index(drop=True)
        if key.nunique() < 2:
            print(f"{gname:14s} {aname:14s} fewer than 2 viable groups -- skipped")
            continue

        cnt = counts[mask]
        X = clr(np.asarray(cnt, dtype=float))
        r = permanova(X, key.to_numpy(), N_PERMUTATIONS, RANDOM_STATE)

        # Descriptive: group-level overdispersion on the pooled counts.
        gc = pd.DataFrame(cnt).groupby(key.to_numpy()).sum().to_numpy()
        d  = overdispersion(gc, N_PERMUTATIONS, RANDOM_STATE)

        verdict = "PASS" if r["eta_excess"] > ETA_EXCESS_FLOOR else "fail"
        results.append({
            "tier": tier, "alphabet": aname, "analysis": "between-group",
            "grouping": gname, "min_reviews": BRAND_MIN,
            "n_groups": r["n_groups"], "n_reviews": r["n_reviews"],
            "statistic": "eta_excess", "value": round(r["eta"], 4),
            "null_value": round(r["eta_null"], 4),
            "excess": round(r["eta_excess"], 4),
            "max_true_sd_pp": round(d["max_true_sd_pp"], 2),
            "group_overdisp": round(d["overdisp_median"], 3),
            "floor": ETA_EXCESS_FLOOR, "verdict": verdict,
        })
        print(f"tier {tier}  {gname:14s} {aname:14s} {r['n_groups']:5d} groups  "
              f"eta {r['eta']:.4f}  null {r['eta_null']:.4f}  "
              f"excess {r['eta_excess']:+.4f}  [{verdict}]")
    print()

# Tier 2: the time-resolved brand tests.
for tier, gname, keys, min_reviews in TIME_GROUPINGS:
    for aname, _ in ALPHABETS:
        counts, meta = DATA[aname]["counts"], DATA[aname]["meta"]
        key  = group_key(meta, keys)
        mask = apply_min(key, min_reviews)
        if mask.sum() == 0 or key[mask].nunique() < 2:
            print(f"{gname:14s} {aname:14s} fewer than 2 viable groups -- skipped")
            continue

        X = clr(np.asarray(counts[mask], dtype=float))
        r = permanova(X, key[mask].to_numpy(), N_PERMUTATIONS, RANDOM_STATE)

        # Descriptive: group-level overdispersion on the pooled counts.
        gc = pd.DataFrame(counts[mask]).groupby(key[mask].to_numpy()).sum().to_numpy()
        d  = overdispersion(gc, N_PERMUTATIONS, RANDOM_STATE)

        verdict = "PASS" if r["eta_excess"] > ETA_EXCESS_FLOOR else "fail"
        results.append({
            "tier": tier, "alphabet": aname, "analysis": "between-group",
            "grouping": gname, "min_reviews": min_reviews,
            "n_groups": r["n_groups"], "n_reviews": r["n_reviews"],
            "statistic": "eta_excess", "value": round(r["eta"], 4),
            "null_value": round(r["eta_null"], 4),
            "excess": round(r["eta_excess"], 4),
            "max_true_sd_pp": round(d["max_true_sd_pp"], 2),
            "group_overdisp": round(d["overdisp_median"], 3),
            "floor": ETA_EXCESS_FLOOR, "verdict": verdict,
        })
        print(f"tier {tier}  {gname:14s} {aname:14s} {r['n_groups']:5d} groups  "
              f"eta {r['eta']:.4f}  null {r['eta_null']:.4f}  "
              f"excess {r['eta_excess']:+.4f}  [{verdict}]")
    print()

tier 1  year           Cluster_30        16 groups  eta 0.0375  null 0.0080  excess +0.0295  [PASS]
tier 1  year           MR                16 groups  eta 0.0180  null 0.0080  excess +0.0101  [fail]
tier 1  year           Agency_6          16 groups  eta 0.0215  null 0.0080  excess +0.0136  [fail]

tier 1  author         Cluster_30        36 groups  eta 0.0521  null 0.0186  excess +0.0335  [PASS]
tier 1  author         MR                36 groups  eta 0.0407  null 0.0187  excess +0.0219  [PASS]
tier 1  author         Agency_6          36 groups  eta 0.0352  null 0.0187  excess +0.0165  [fail]

tier 1  brand          Cluster_30        65 groups  eta 0.0458  null 0.0340  excess +0.0119  [fail]
tier 1  brand          MR                65 groups  eta 0.0487  null 0.0341  excess +0.0146  [fail]
tier 1  brand          Agency_6          65 groups  eta 0.0528  null 0.0341  excess +0.0187  [fail]

tier 2  brand x 3yr    Cluster_30       518 groups  eta 0.1914  null 0.1650  excess +0.0264  [PAS

## 3. Tier 3: within a brand

In [6]:
# Cell 7: Tier 3a: within-brand consistency

for aname, _ in ALPHABETS:
    counts, meta = DATA[aname]["counts"], DATA[aname]["meta"]
    X = clr(np.asarray(counts, dtype=float))

    brand = meta["brand"].to_numpy()
    kept  = [b for b in sorted(PANEL_BRANDS) if (brand == b).any()]

    rng = np.random.default_rng(RANDOM_STATE)
    null_by_size = {}                      # null depends on set size only
    ratios = []
    for b in kept:
        rows = np.where(brand == b)[0]
        n = len(rows)
        obs = float(pdist(X[rows]).mean())
        if n not in null_by_size:
            draws = [float(pdist(X[rng.choice(len(X), n, replace=False)]).mean())
                     for _ in range(N_PERMUTATIONS)]
            null_by_size[n] = (float(np.mean(draws)), float(np.std(draws, ddof=1)))
        nm, nsd = null_by_size[n]
        ratios.append({"brand": b, "n": n, "obs": obs, "null": nm,
                       "ratio": obs / nm, "z": (obs - nm) / nsd if nsd else np.nan})

    cons = pd.DataFrame(ratios)
    median_ratio = float(cons["ratio"].median())
    tighter = int((cons["z"] < -1.96).sum())

    # Excess framed like eta: how much tighter than chance, as a share of the null.
    excess = 1.0 - median_ratio
    verdict = "PASS" if excess > ETA_EXCESS_FLOOR else "fail"
    results.append({
        "tier": 3, "alphabet": aname, "analysis": "within-brand consistency",
        "grouping": "brand", "min_reviews": BRAND_MIN,
        "n_groups": len(cons), "n_reviews": int(cons["n"].sum()),
        "statistic": "1 - dispersion_ratio", "value": round(median_ratio, 4),
        "null_value": 1.0, "excess": round(excess, 4),
        "max_true_sd_pp": np.nan, "floor": ETA_EXCESS_FLOOR, "verdict": verdict,
    })
    print(f"{aname:14s} {len(cons):3d} brands   median dispersion ratio "
          f"{median_ratio:.4f}   tighter than chance (z<-1.96): {tighter}   [{verdict}]")


Cluster_30      65 brands   median dispersion ratio 1.0171   tighter than chance (z<-1.96): 3   [fail]
MR              65 brands   median dispersion ratio 0.9990   tighter than chance (z<-1.96): 0   [fail]
Agency_6        65 brands   median dispersion ratio 0.9804   tighter than chance (z<-1.96): 5   [fail]


In [7]:
# Cell 8: Tier 3b:  within-brand temporal drift (panel brands)

for aname, _ in ALPHABETS:
    counts, meta = DATA[aname]["counts"], DATA[aname]["meta"]
    X = clr(np.asarray(counts, dtype=float))

    brand = meta["brand"].to_numpy()
    years = pd.to_numeric(meta["year"], errors="coerce").to_numpy(dtype=float)
    kept  = [b for b in sorted(PANEL_BRANDS) if (brand == b).any()]

    rng = np.random.default_rng(RANDOM_STATE)
    rows_out = []
    for b in kept:
        rows = np.where((brand == b) & ~np.isnan(years))[0]
        if len(rows) < 3:
            continue
        d  = pdist(X[rows])
        dt = pdist(years[rows].reshape(-1, 1))     # |year_i - year_j|
        if np.allclose(dt, dt[0]):
            continue
        obs = scipy.stats.spearmanr(dt, d).statistic
        null = np.empty(N_PERMUTATIONS)
        for i in range(N_PERMUTATIONS):
            yp = rng.permutation(years[rows])
            null[i] = scipy.stats.spearmanr(pdist(yp.reshape(-1, 1)), d).statistic
        rows_out.append({"brand": b, "n": len(rows), "rho": obs,
                         "null": float(np.mean(null)),
                         "excess": obs - float(np.mean(null))})

    drift = pd.DataFrame(rows_out)
    median_excess = float(drift["excess"].median())
    positive = int((drift["excess"] > 0).sum())

    verdict = "PASS" if median_excess > ETA_EXCESS_FLOOR else "fail"
    results.append({
        "tier": 3, "alphabet": aname, "analysis": "within-brand drift",
        "grouping": "brand", "min_reviews": BRAND_MIN,
        "n_groups": len(drift), "n_reviews": int(drift["n"].sum()),
        "statistic": "spearman(time, distance)",
        "value": round(float(drift["rho"].median()), 4),
        "null_value": round(float(drift["null"].median()), 4),
        "excess": round(median_excess, 4),
        "max_true_sd_pp": np.nan, "floor": ETA_EXCESS_FLOOR, "verdict": verdict,
    })
    print(f"{aname:14s} {len(drift):3d} brands   median rho {drift['rho'].median():+.4f}   "
          f"excess {median_excess:+.4f}   brands with positive excess: "
          f"{positive}/{len(drift)}   [{verdict}]")

print()
print("Positive means a brand's composition moves with time within its own history.")
print("Read the coefficient: rho = 0.1 is a 1% shared-variance relationship.")

Cluster_30      65 brands   median rho +0.0649   excess +0.0643   brands with positive excess: 51/65   [PASS]
MR              65 brands   median rho -0.0065   excess -0.0051   brands with positive excess: 29/65   [fail]
Agency_6        65 brands   median rho -0.0009   excess -0.0014   brands with positive excess: 31/65   [fail]

Positive means a brand's composition moves with time within its own history.
Read the coefficient: rho = 0.1 is a 1% shared-variance relationship.


## 4. Summary

In [8]:
# Cell 9: Assemble and append to the stacked results file

new = pd.DataFrame(results)
new["floor_in_force"] = new["floor"]
new["seed"] = RANDOM_STATE
new["n_permutations"] = N_PERMUTATIONS
new["alphabet_source"] = SEQ_PATH.name
new["run_utc"] = RUN_UTC

KEY = ["tier", "alphabet", "analysis", "grouping", "min_reviews"]

if RESULTS_PATH.exists():
    old = pd.read_csv(RESULTS_PATH)
    merged = pd.concat([old.merge(new[KEY], on=KEY, how="left", indicator=True)
                          .query("_merge == 'left_only'").drop(columns="_merge"),
                        new], ignore_index=True)
    print(f"Appended; {len(old)} existing rows, {len(new)} from this run "
          f"({len(old) - (len(merged) - len(new))} replaced)")
else:
    merged = new
    print(f"Created {RESULTS_PATH} with {len(new)} rows")

merged = merged.sort_values(["tier", "grouping", "alphabet"]).reset_index(drop=True)
merged.to_csv(RESULTS_PATH, index=False)
print(f"Saved {RESULTS_PATH}  ({len(merged)} rows total)")

Appended; 57 existing rows, 21 from this run (21 replaced)
Saved outputs_event_chains/experiment/composition_scale.csv  (57 rows total)


In [9]:
# Cell 10: results table

show = ["tier", "analysis", "grouping", "alphabet", "n_groups", "n_reviews",
        "value", "null_value", "excess", "max_true_sd_pp", "verdict"]
print(new[show].to_string(index=False))
print()
print(f"FLOORS: Tier 0 overdispersion > {OVERDISPERSION_FLOOR};  "
      f"Tiers 1-3 excess > {ETA_EXCESS_FLOOR}")

 tier                 analysis     grouping   alphabet  n_groups  n_reviews   value  null_value  excess  max_true_sd_pp verdict
    1            between-group         year Cluster_30        16       1883  0.0375      0.0080  0.0295            2.00    PASS
    1            between-group         year         MR        16       1883  0.0180      0.0080  0.0101            0.99    fail
    1            between-group         year   Agency_6        16       1883  0.0215      0.0080  0.0136            1.88    fail
    1            between-group       author Cluster_30        36       1883  0.0521      0.0186  0.0335            3.32    PASS
    1            between-group       author         MR        36       1883  0.0407      0.0187  0.0219            5.59    PASS
    1            between-group       author   Agency_6        36       1883  0.0352      0.0187  0.0165            3.92    fail
    1            between-group        brand Cluster_30        65       1883  0.0458      0.0340  0.0119 

In [10]:
# Cell 11: The comparison that decides the framing

# Rule 2 of the pre-registration: a brand result only counts if it also beats author.

between = new[new["analysis"] == "between-group"]
for aname, _ in ALPHABETS:
    sub = between[between["alphabet"] == aname].set_index("grouping")["excess"]
    brand  = sub.get("brand")
    author = sub.get("author")
    b3     = sub.get("brand x 3yr")
    print(f"{aname}:")
    print(f"   author       {author:+.4f}")
    print(f"   brand        {brand:+.4f}")
    print(f"   brand x 3yr  {b3:+.4f}")
    if author >= brand:
        print("   -> author explains at least as much as brand: composition tracks the")
        print("      critic's writing style, not the collection.")
    else:
        print("   -> brand outperforms author.")
    print()

Cluster_30:
   author       +0.0335
   brand        +0.0119
   brand x 3yr  +0.0264
   -> author explains at least as much as brand: composition tracks the
      critic's writing style, not the collection.

MR:
   author       +0.0219
   brand        +0.0146
   brand x 3yr  +0.0142
   -> author explains at least as much as brand: composition tracks the
      critic's writing style, not the collection.

Agency_6:
   author       +0.0165
   brand        +0.0187
   brand x 3yr  +0.0227
   -> brand outperforms author.



## 5. Robustness: the smoothing constant

In [11]:
# Cell 12: Does the smoothing constant change the brand-vs-author verdict?

SMOOTHING_GRID = [0.1, 0.5, 1.0, 2.0]

rows = []
for aname, _ in ALPHABETS:
    counts, meta = DATA[aname]["counts"], DATA[aname]["meta"]
    mask = panel_mask(meta)          # brand and author both on the same panel subset
    for s in SMOOTHING_GRID:
        out = {}
        X = clr(np.asarray(counts[mask], dtype=float), s)
        for gname, keys in [("brand", ["brand"]), ("author", ["author"])]:
            key = group_key(meta[mask], keys).reset_index(drop=True)
            out[gname] = permanova(X, key.to_numpy(),
                                   N_PERMUTATIONS, RANDOM_STATE)["eta_excess"]
        rows.append({"alphabet": aname, "smoothing": s,
                     "brand": round(out["brand"], 4),
                     "author": round(out["author"], 4),
                     "brand/author": round(out["brand"] / out["author"], 3),
                     "brand_beats_author": out["brand"] > out["author"]})

sens = pd.DataFrame(rows)
print(sens.to_string(index=False))
print()
if not sens["brand_beats_author"].any():
    print("Brand fails to beat author at every smoothing value tested, at every")
    print("resolution. The verdict does not depend on the constant.")
else:
    print("The verdict CHANGES with the smoothing constant:  report this and do not")
    print("rely on the fine-grained resolution.")

  alphabet  smoothing  brand  author  brand/author  brand_beats_author
Cluster_30        0.1 0.0088  0.0206         0.427               False
Cluster_30        0.5 0.0119  0.0335         0.354               False
Cluster_30        1.0 0.0146  0.0453         0.323               False
Cluster_30        2.0 0.0186  0.0626         0.297               False
        MR        0.1 0.0095  0.0104         0.911               False
        MR        0.5 0.0146  0.0219         0.667               False
        MR        1.0 0.0182  0.0378         0.482               False
        MR        2.0 0.0234  0.0681         0.344               False
  Agency_6        0.1 0.0152  0.0066         2.314                True
  Agency_6        0.5 0.0187  0.0165         1.139                True
  Agency_6        1.0 0.0207  0.0270         0.766               False
  Agency_6        2.0 0.0231  0.0443         0.521               False

The verdict CHANGES with the smoothing constant:  report this and do not
rel

## 6. PERMANOVA assumptions

`permanova()` is distribution-free (no multivariate normality), but three assumptions
still bear on how the verdicts are read.

- **A: homogeneity of multivariate dispersion (PERMDISP / `betadisper`).** PERMANOVA's
  pseudo-F is sensitive to differences in *within-group spread*, not only *centroid
  location*. Because the `eta_excess` statistic subtracts the permutation mean rather
  than thresholding a permutation p-value, heterogeneous dispersion cannot manufacture a
  false pass here: but where a grouping passes, the honest wording is "location
  and/or spread", so the check flags which groupings sit on unequal dispersion. CLR is
  Euclidean, so the group centroid is exactly the group mean and no PCoA is needed.
- **B: pseudo-replication.** Reviews are not iid within a brand: a handful of critics
  write most of a house's coverage, so the free-label null overstates the effective
  sample size (3.4× at the fine alphabet). The author-collapsed test reduces each
  `brand × author` cell to a single mean-CLR point and re-checks whether the brand
  verdict survives de-replication.
- **C: exchangeability / temporal non-independence.** Free label permutation assumes
  reviews within a group are exchangeable. If reviews near in time are also near in
  composition, that is violated and the null is mildly anticonservative. The check
  reports the within-brand Spearman ρ between time gap and compositional distance; a
  large ρ would call for permutations restricted within time (as Tier 3b already does),
  a small one does not move any verdict.

None of the three overturns the Tiers 1–2 conclusions; B and C in fact cut toward the
existing "author, not brand" framing.

In [12]:
# Cell 13: Assumption checks helpers
CHECK_GROUPINGS = [("year", ["year"]), ("author", ["author"]), ("brand", ["brand"])]


def _dist_to_own_centroid(X, labels, G):
    n = X.shape[0]
    M = csr_matrix((np.ones(n), (labels, np.arange(n))), shape=(G, n))
    cnt = np.bincount(labels, minlength=G).astype(float)
    means = M @ X
    means[cnt > 0] /= cnt[cnt > 0, None]
    return np.sqrt(((X - means[labels]) ** 2).sum(axis=1))


def permdisp(X, key_values, n_perm, seed):
    """betadisper analogue: permutation F-test that all groups share one dispersion.
    CLR is Euclidean, so the group centroid is the group mean and no PCoA is needed."""
    labels = pd.Categorical(key_values).codes.astype(np.int64)
    G = int(labels.max()) + 1

    def F_of(lab):
        d = _dist_to_own_centroid(X, lab, G)
        gm = np.array([d[lab == g].mean() for g in range(G)])
        cnt = np.bincount(lab, minlength=G).astype(float)
        ssb_d = (cnt * (gm - d.mean()) ** 2).sum()
        ssw_d = ((d - gm[lab]) ** 2).sum()
        return (ssb_d / (G - 1)) / (ssw_d / (len(d) - G)), gm

    F, gm = F_of(labels)
    rng = np.random.default_rng(seed)
    ge = sum(F_of(rng.permutation(labels))[0] >= F for _ in range(n_perm))
    return {"G": G, "F": F, "p": (ge + 1) / (n_perm + 1),
            "disp_ratio": gm.max() / gm.min(), "disp_cv": gm.std() / gm.mean()}

In [13]:
# Check A: Dispersion homogeneity

print("A: DISPERSION HOMOGENEITY  (all on the panel)")
print(f"   {'alphabet':14s} {'grouping':8s} {'G':>4s} {'F':>7s} {'p':>6s} "
      f"{'ratio':>6s} {'cv':>6s}  read as")
for aname, _ in ALPHABETS:
    X = clr(np.asarray(DATA[aname]["counts"], dtype=float))
    meta = DATA[aname]["meta"]
    mask = panel_mask(meta)
    for gname, keys in CHECK_GROUPINGS:
        key = group_key(meta[mask], keys).reset_index(drop=True)
        r = permdisp(X[mask], key.to_numpy(), N_PERMUTATIONS, RANDOM_STATE)
        read = "location and/or spread" if r["p"] < 0.05 else "location"
        print(f"   {aname:14s} {gname:8s} {r['G']:4d} {r['F']:7.2f} {r['p']:6.3f} "
              f"{r['disp_ratio']:6.2f} {r['disp_cv']:6.3f}  {read}")

A: DISPERSION HOMOGENEITY  (all on the panel)
   alphabet       grouping    G       F      p  ratio     cv  read as
   Cluster_30     year       16   23.47  0.005   1.44  0.084  location and/or spread
   Cluster_30     author     36   20.28  0.005    inf  0.473  location and/or spread


/var/folders/_p/f1mg9b290kx6bf1ytjkd60qm0000gn/T/ipykernel_15159/2729241869.py:32: RuntimeWarning: divide by zero encountered in scalar divide
  "disp_ratio": gm.max() / gm.min(), "disp_cv": gm.std() / gm.mean()}


   Cluster_30     brand      65    3.12  0.005   1.26  0.053  location and/or spread
   MR             year       16    1.82  0.030   1.25  0.051  location and/or spread
   MR             author     36    2.34  0.065    inf  0.514  location


/var/folders/_p/f1mg9b290kx6bf1ytjkd60qm0000gn/T/ipykernel_15159/2729241869.py:32: RuntimeWarning: divide by zero encountered in scalar divide
  "disp_ratio": gm.max() / gm.min(), "disp_cv": gm.std() / gm.mean()}


   MR             brand      65    0.83  0.886   1.36  0.075  location
   Agency_6       year       16    3.83  0.005   1.23  0.059  location and/or spread
   Agency_6       author     36    3.69  0.005    inf  0.499  location and/or spread
   Agency_6       brand      65    1.32  0.095   1.53  0.073  location


/var/folders/_p/f1mg9b290kx6bf1ytjkd60qm0000gn/T/ipykernel_15159/2729241869.py:32: RuntimeWarning: divide by zero encountered in scalar divide
  "disp_ratio": gm.max() / gm.min(), "disp_cv": gm.std() / gm.mean()}


In [14]:
# Check B: Pseudo-replication

FINE = "Cluster_30"   # the fine 30-type alphabet
meta = DATA[FINE]["meta"]
X = clr(np.asarray(DATA[FINE]["counts"], dtype=float))
mask = panel_mask(meta)
sub, Xsub = meta[mask].reset_index(drop=True), X[mask]

cells = pd.Categorical(group_key(sub, ["brand", "author"]))
C = len(cells.categories)
Msel = csr_matrix((np.ones(len(cells.codes)), (cells.codes, np.arange(len(cells.codes)))),
                  shape=(C, len(cells.codes)))
cell_mean = np.asarray(Msel @ Xsub) / np.asarray(Msel.sum(axis=1)).ravel()[:, None]
cell_brand = pd.Series(cells.categories).str.split(r" \| ").str[0].to_numpy()
bc = pd.Series(cell_brand).value_counts()
km = pd.Series(cell_brand).isin(bc[bc >= 5].index).to_numpy()

r_raw = permanova(Xsub, group_key(sub, ["brand"]).to_numpy(), N_PERMUTATIONS, RANDOM_STATE)
r_col = permanova(cell_mean[km], cell_brand[km], N_PERMUTATIONS, RANDOM_STATE)
print("B  PSEUDO-REPLICATION (brand, fine alphabet, panel)")
print(f"   replication factor       : {mask.sum() / C:.2f}x  "
      f"({mask.sum():,} reviews / {C:,} brand-author cells)")
print(f"   review-level    excess   : {r_raw['eta_excess']:+.4f}   (floor {ETA_EXCESS_FLOOR})")
print(f"   author-collapsed excess  : {r_col['eta_excess']:+.4f}\n")

B  PSEUDO-REPLICATION (brand, fine alphabet, panel)
   replication factor       : 4.50x  (1,883 reviews / 418 brand-author cells)
   review-level    excess   : +0.0119   (floor 0.02)
   author-collapsed excess  : -0.0006



In [15]:
# C  Temporal non-independence within brands.  (sub is already the panel)
years = pd.to_numeric(sub["year"], errors="coerce").to_numpy(dtype=float)
brand_arr = sub["brand"].to_numpy()
rhos = []
for b in pd.unique(brand_arr):
    rows = np.where((brand_arr == b) & ~np.isnan(years))[0]
    if len(rows) < 3:
        continue
    dt = pdist(years[rows].reshape(-1, 1))
    if np.allclose(dt, dt[0]):
        continue
    rhos.append(scipy.stats.spearmanr(dt, pdist(Xsub[rows])).statistic)
rhos = np.array(rhos)
print("C  TEMPORAL NON-INDEPENDENCE within brands   rho(|dt|, composition distance)")
print(f"   brands tested            : {len(rhos)}")
print(f"   median within-brand rho  : {np.median(rhos):+.4f}   "
      f"({(rhos > 0).mean() * 100:.0f}% positive, ~{np.median(rhos) ** 2 * 100:.1f}% shared var)")

C  TEMPORAL NON-INDEPENDENCE within brands   rho(|dt|, composition distance)
   brands tested            : 65
   median within-brand rho  : +0.0649   (75% positive, ~0.4% shared var)
